https://ujwal-s-r.github.io/CUDA-triton-GPU-kernals/0_what_is_GPU/GPU_visualize/index.html

# Foundations — Building the GPU from the Ground Up

## 1. The Core Problem: Why Can’t a CPU Just Run Neural Networks Fast?

To understand a GPU, you first need to understand the ALU and why a CPU runs out of steam.

### What is an ALU?

An ALU (Arithmetic Logic Unit) is a physical circuit made of transistors that takes numbers as inputs and performs one basic math operation such as addition, subtraction, or multiplication.

If you want to compute:

$$c = a + b$$

an ALU performs that work directly.

### How a CPU Uses an ALU

A modern CPU is built for flexibility, not raw arithmetic throughput. It must:

- run an operating system,
- handle user input and interrupts,
- execute branching logic such as if/else statements.

Because a CPU must switch between many different task types, most of its silicon is devoted to:

- branch predictors,
- out-of-order execution logic,
- large caches.

As a result, a high-end CPU has relatively few ALUs compared to the number of control structures it needs.

### Why This Fails for Deep Learning

Deep learning workloads are mostly repetitive math. A matrix multiplication can be expressed as:

$$Y = XW$$

and internally as:

$$\text{multiply} \rightarrow \text{add to a running sum} \rightarrow \text{repeat billions of times}$$

If a CPU has only a few dozen ALUs, it cannot keep up with the enormous demand for parallel arithmetic.

---

## 2. The GPU Design Philosophy — Trading Control for Muscle

GPU architects asked a simple question:

> What if we remove much of the complex control logic and use that silicon to pack thousands of ALUs onto one chip?

That led to a throughput-optimized design:

```text
CPU CORE (Latency-Optimized)
┌────────────────────────────────────────┐
│ [ Large Branch Predictor / Decoder ]   │
│ [ Out-of-Order Execution Reorder Buffer]│
│ [ Giant L1/L2 Caches ]                 │
│ ┌────────────────────────────────────┐ │
│ │ ALU (Only a tiny fraction of area) │ │
│ └────────────────────────────────────┘ │
└────────────────────────────────────────┘
```

```text
GPU SILICON DIE (Throughput-Optimized)
┌────────────────────────────────────────────────────────┐
│ [ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU]... │
│ [ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU]... │
│ [ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU][ALU]... │
│ (Thousands of simple ALUs doing math in parallel)      │
│ ┌──────────────────────┐   ┌─────────────────────────┐ │
│ │ Small Shared Caches  │   │ Minimal Control Logic   │ │
│ └──────────────────────┘   └─────────────────────────┘ │
└────────────────────────────────────────────────────────┘
```

This creates a new challenge: how do you control thousands of ALUs without building thousands of instruction decoders?

That is where the warp comes in.


## What Exactly Is a “Warp”? (SIMT Execution)

Imagine you have 32 workers and need them to add 32 pairs of numbers.

You have two design choices:

### Option A: The CPU Approach

Give every worker their own personal manager with a megaphone. That requires 32 managers for 32 workers, which is expensive in hardware.

### Option B: The GPU Approach

Give one manager a megaphone and have that manager instruct all 32 workers at the same time:

> “Everyone, take your number from Register A, add it to Register B, and save it to Register C.”

### Definition: A Warp

A warp is a hardware group of 32 execution lanes that execute the same instruction at the same time.

```text
ONE INSTRUCTION DISPATCHED: "ADD"
                                │
        ┌───────────────────────┴───────────────────────┐
        ▼                                               ▼
   [ Lane 0 ]                                      [ Lane 31 ]
┌────────────────┐                              ┌────────────────┐
│ Thread 0       │                              │ Thread 31      │
│ Reg A: 5       │                              │ Reg A: 12      │
│ Reg B: 3       │                              │ Reg B: 8       │
│ ALU: 5 + 3 = 8 │                              │ ALU: 12 + 8 =20│
│ Reg C: 8       │                              │ Reg C: 20      │
└────────────────┘                              └────────────────┘
```

- Lane: one hardware slot inside the warp.
- Thread: the software view of that lane, with its own private data and ID.

### SIMT (Single Instruction, Multiple Threads)

SIMT is the execution model where one instruction is decoded and then applied to many independent threads across different data.


## The Problem: What Happens If Code Branches? (Warp Divergence)

Suppose 32 threads in a warp run this code:

```python
if thread_id < 16:
    x = x + 1
else:
    x = x * 2
```

Because the warp has only one instruction decoder, it cannot run both branches at the same time.

### What the Hardware Does

- In clock cycle 1, lanes 0–15 execute the addition while lanes 16–31 stay idle.
- In clock cycle 2, lanes 16–31 execute the multiplication while lanes 0–15 stay idle.

```text
Cycle 1: [T0..T15: ACTIVE (+)]  [T16..T31: IDLE]
Cycle 2: [T0..T15: IDLE]       [T16..T31: ACTIVE (*)]
```

This is called warp divergence, and it is a major reason that poorly structured GPU code can run slowly.

## 4. The Building Block of the GPU — The Streaming Multiprocessor (SM)

Once you understand warps, the next question is where they live on the chip. They live inside a Streaming Multiprocessor (SM), which is a self-contained execution engine.

```text
┌────────────────────────────────────────────────────────────────────────┐
│ STREAMING MULTIPROCESSOR (SM)                                         │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ 1. Warp Schedulers: Track active warps and issue instructions   │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ 2. Register File: Ultra-fast SRAM for thread-private state     │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ 3. Execution Units: CUDA Cores, SFUs, Tensor Cores             │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ 4. Shared Memory / L1 Cache: Fast scratchpad for collaboration │  │
│  └──────────────────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────────────────┘
```

### 1. Warp Scheduler

An SM can hold multiple warps at the same time. When one warp is waiting for memory, the scheduler can switch to another warp that is ready to execute.

### 2. Register File

This is the fastest memory on the chip and sits very close to the ALUs. Each thread gets its own private register slice.

### 3. Shared Memory / L1 Cache

This on-chip SRAM is shared by threads inside a thread block, allowing them to cooperate efficiently without repeatedly reading from slow global memory.

---

## 5. What Is a CUDA Core vs. a Tensor Core?

Inside an SM, there are two major types of compute units.

### CUDA Core

A CUDA core performs scalar arithmetic such as:

$$d = (a \times b) + c$$

This is a single arithmetic operation for one value.

### Tensor Core

A tensor core performs dense matrix operations directly in hardware, such as:

$$D_{16\times16} = (A_{16\times16} \times B_{16\times16}) + C_{16\times16}$$

This is why tensor cores are so important for AI workloads: matrix multiply is the core operation in transformer models and large language models.


### The Physical Silicon Floorplan of an NVIDIA GPU

When you hold a data-center accelerator (such as an NVIDIA A100 or H100), you are looking at a system built hierarchically from the physical board down to microscopic circuits:

Here is the physical hierarchy of a modern NVIDIA GPU:

```text
[ GPU Board / Accelerator Package ]
     │
     ├── High-Bandwidth Memory (HBM3 / HBM3e)
     │
     └── GPU Die (Silicon ASIC)
          │
          └── [ L2 Cache & Interconnect Fabric (NVLink/PCIe) ]
               │
               ├── GPC (GPU Processing Cluster) 0
               ├── GPC 1
               │    ├── TPC (Texture Processing Cluster) 0
               │    ├── TPC 1
               │    │    ├── SM (Streaming Multiprocessor) 0
               │    │    └── SM (Streaming Multiprocessor) 1
               │    │         │
               │    │         ├── Warp Schedulers & Instruction Dispatchers
               │    │         ├── Register File (Massive fast storage)
               │    │         ├── L1 Instruction Cache / L1 Data Cache & Shared Memory (SRAM)
               │    │         ├── Standard CUDA Cores (FP32, FP64, INT32 ALUs)
               │    │         └── Tensor Cores (Matrix Multiply-Accumulate Units)
               │    └── ...
               └── ...
```


**Step-by-Step Breakdown of the Hierarchy:**

- **Board / Package Level:** The silicon die sits on a substrate alongside stacks of HBM (High-Bandwidth Memory) or discrete GDDR chips, connected via ultra-fast interfaces.
- **GPU Processing Cluster (GPC):** The GPU die is partitioned into several large physical islands called GPCs. A GPC houses dedicated rasterization hardware (historical graphics origin) and multiple TPCs.
- **Texture Processing Cluster (TPC):** An intermediate grouping within a GPC that typically holds two Streaming Multiprocessors.
- **Streaming Multiprocessor (SM):** This is the fundamental building block of GPU compute. Everything you run in PyTorch, Triton, or CUDA ultimately schedules and executes inside an SM. An enterprise GPU contains between $ and +$ SMs.

### Inside the Streaming Multiprocessor (SM)

```text
┌────────────────────────────────────────────────────────────────────────┐
│ STREAMING MULTIPROCESSOR (SM)                                         │
│                                                                      │
│  ┌────────────────────────┐  ┌──────────────────────────────────────┐  │
│  │ L1 Instruction Cache   │  │ Register File (e.g., 256 KB SRAM)    │  │
│  └────────────────────────┘  └──────────────────────────────────────┘  │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ 4 Processing Blocks (Sub-Cores), each containing:              │  │
│  │  • Warp Scheduler & Dispatch Unit                              │  │
│  │  • Integer ALUs (INT32)                                         │  │
│  │  • Floating Point ALUs (FP32, FP64)                             │  │
│  │  • Special Function Units (SFUs: sin, cos, exp, rsqrt)          │  │
│  │  • Tensor Core(s) (Dense Matrix Multiply Units)                 │  │
│  │  • Load/Store Units (LSU)                                       │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ Unified Shared Memory / L1 Data Cache (e.g., 128 KB–228 KB SRAM)│ │
│  └──────────────────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────────────────┘
```

## The Transistor Tradeoff

### Why CPUs Waste Silicon on Control

CPUs are designed for low latency. To keep pipelines full, they use features such as:

- branch predictors,
- speculative execution,
- reorder buffers,
- register renaming.

These structures improve single-thread performance, but they consume a lot of silicon.

### Why GPUs Throw This Away

GPUs are optimized for throughput. For deep learning workloads, the control flow is often simple and repetitive. That makes it more effective to dedicate silicon to many ALUs rather than to complex control logic.

### How GPUs Hide Memory Latency

When a GPU thread waits for memory, the hardware does not speculate aggressively. Instead, it switches to another warp that is ready to execute. This keeps the chip busy even while one warp is stalled on memory access.

```text
+-------------------------------------------------------------------------+
| ACCELERATOR PACKAGE / BOARD                                             |
|                                                                         |
|  +-------------+  +-------------+  +-------------+  +-------------+     |
|  |  HBM3/HBM3e |  |  HBM3/HBM3e |  |  HBM3/HBM3e |  |  HBM3/HBM3e |     |
|  |  Stack (16G)|  |  Stack (16G)|  |  Stack (16G)|  |  Stack (16G)|     |
|  +------+------+  +------+------+  +------+------+  +------+------+     |
|         |                |                |                |            |
|  =======+================+================+================+==========  |
|         |           Ultra-Wide Silicon Interposer / Substrate           |
|  =======+================+================+================+==========  |
|         |                |                |                |            |
|  +------+----------------+----------------+----------------+---------+  |
|  | SILICON ASIC (GPU DIE)                                             |
|  |                                                                     |
|  |   +------------------------------------------------------------+   |
|  |   | 50MB–60MB L2 Cache (SRAM, Crossbar Interconnect)          |   |
|  |   +------------------------------------------------------------+   |
|  |   | PCIe Gen5 Host Controller / NVLink 4/5 Network Interfaces |   |
|  |   +------------------------------------------------------------+   |
|  |                                                                     |
|  |   +-----------------------+     +-----------------------+         |
|  |   | GPC 0 (GPU Cluster)   | ... | GPC 7 (GPU Cluster)   |         |
|  |   |   +-----------------+ |     |                       |         |
|  |   |   | TPC 0           | |     |                       |         |
|  |   |   |   +-----------+ | |     |                       |         |
|  |   |   |   | SM 0      | | |     |                       |         |
|  |   |   |   | SM 1      | | |     |                       |         |
|  |   |   |   +-----------+ | |     |                       |         |
|  |   |   +-----------------+ |     |                       |         |
|  |   +-----------------------+     +-----------------------+         |
|  +-------------------------------------------------------------------+  |
+-------------------------------------------------------------------------+
```

## 1. The GPC (GPU Processing Cluster)

```text
┌─────────────────────────────────────────────────────────────────────────┐
│ GPU PROCESSING CLUSTER (GPC)                                            │
│                                                                         │
│  ┌─────────────────────────────────┐ ┌────────────────────────────────┐ │
│  │ Raster Engine / Work Distributor│ │ Dedicated L1.5 / Crossbar Links│ │
│  │ (Geometry, Tessellation Setup)  │ │ (Routes traffic to L2 Cache)   │ │
│  └─────────────────────────────────┘ └────────────────────────────────┘ │
│                                                                         │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐ │
│  │    TPC 0     │  │    TPC 1     │  │    TPC 2     │  │ ... TPC N    │ │
│  │ ┌────┐┌────┐ │  │ ┌────┐┌────┐ │  │ ┌────┐┌────┐ │  │ ┌────┐┌────┐ │ │
│  │ │SM 0││SM 1│ │  │ │SM 2││SM 3│ │  │ │SM 4││SM 5│ │  │ │SM..││SM..│ │ │
│  │ └────┘└────┘ │  │ └────┘└────┘ │  │ └────┘└────┘ │  │ └────┘└────┘ │ │
│  └──────────────┘  └──────────────┘  └──────────────┘  └──────────────┘ │
└─────────────────────────────────────────────────────────────────────────┘
```

### Hierarchical Work Distribution

When PyTorch launches a grid of 100,000 thread blocks, the chip-level scheduler does not send work directly to each SM. Instead, it dispatches coarse batches of work to GPCs, and each GPC routes work into its child TPCs and SMs.

## 2. The TPC (Texture Processing Cluster)

A TPC sits inside a GPC and acts as a bridge between the GPC and the SMs. In modern CUDA architectures, a TPC typically contains two SMs plus shared auxiliary pipelines.

```text
┌──────────────────────────────────────────────────────────────┐
│ TEXTURE PROCESSING CLUSTER (TPC)                             │
│                                                              │
│  ┌────────────────────────────────────────────────────────┐  │
│  │ PolyMorph Engine / Ray Tracing (RT) Core Unit (Shared) │  │
│  └────────────────────────────────────────────────────────┘  │
│                                                              │
│  ┌─────────────────────────┐    ┌─────────────────────────┐  │
│  │       SM 0              │    │       SM 1              │  │
│  │  - Warp Schedulers      │    │  - Warp Schedulers      │  │
│  │  - Register File        │    │  - Register File        │  │
│  │  - Tensor Cores         │    │  - Tensor Cores         │  │
│  │  - CUDA Cores           │    │  - CUDA Cores           │  │
│  │  - L1 / Shared Memory   │    │  - L1 / Shared Memory   │  │
│  └────────────┬────────────┘    └────────────┬────────────┘  │
│               │                              │               │
│               └──────────────┬───────────────┘               │
│                              │                               │
│               ┌──────────────▼──────────────┐                │
│               │  Shared Texture / L1.5 Unit │                │
│               └──────────────┬──────────────┘                │
│                              │                               │
└──────────────────────────────┼───────────────────────────────┘
                               │
               (To GPC Interconnect / L2 Crossbar)
```


## 3. Detailed Anatomy of a Streaming Multiprocessor (SM)

An SM is an independent execution engine. In architectures such as H100 or A100, each SM is internally split into several sub-cores or processing blocks.

```text
+----------------------------------------------------------------------------------------------------+
| STREAMING MULTIPROCESSOR (SM)                                                                      |
|                                                                                                    |
|  +----------------------------------------------------------------------------------------------+  |
|  | L1 Instruction Cache (Shared across the 4 Sub-Cores)                                         |  |
|  +----------------------------------------------------------------------------------------------+  |
|                                                                                                    |
|  +---------------------------+ +---------------------------+ +----------------------------------+  |
|  | SUB-CORE 0                | | SUB-CORE 1                | | SUB-CORE 2 / 3                   |  |
|  |                           | |                           | |                                  |  |
|  | [ Warp Scheduler ]        | | [ Warp Scheduler ]        | | (Identical layout)               |  |
|  | [ Instruction Dispatch ]  | | [ Instruction Dispatch ]  | |                                  |  |
|  |                           | |                           | |                                  |  |
|  | [ Register File: 64 KB ]  | | [ Register File: 64 KB ]  | |                                  |  |
|  | (16,384 x 32-bit reg)     | | (16,384 x 32-bit reg)     | |                                  |  |
|  |                           | |                           | |                                  |  |
|  | [ 16 x FP32 Cores ]       | | [ 16 x FP32 Cores ]       | |                                  |  |
|  | [ 16 x INT32 Cores ]      | | [ 16 x INT32 Cores ]      | |                                  |  |
|  | [ 8 x FP64 Cores ]        | | [ 8 x FP64 Cores ]      | |                                  |  |
|  | [ 4 x SFUs ]              | | [ 4 x SFUs ]              | |                                  |  |
|  | [ 1 x 4th-Gen Tensor Core]| | [ 1 x 4th-Gen Tensor Core]| |                                  |  |
|  | [ 8 x Load/Store Units ]  | | [ 8 x Load/Store Units ]  | |                                  |  |
|  +---------------------------+ +---------------------------+ +----------------------------------+  |
|                                                                                                    |
|  +----------------------------------------------------------------------------------------------+  |
|  | COMBINED SHARED MEMORY / L1 DATA CACHE (Configurable SRAM: up to 228 KB per SM)           |  |
|  | - 32 independent banks (4 bytes wide per bank = 128 bytes/cycle throughput)                 |  |
|  +----------------------------------------------------------------------------------------------+  |
|  | TENSOR MEMORY ACCELERATOR (TMA) ENGINE (Asynchronous hardware copy: Global Memory <-> SRAM) |  |
|  +----------------------------------------------------------------------------------------------+  |
+----------------------------------------------------------------------------------------------------+
```


+----------------------------------------------------------------------------------------------------+
| STREAMING MULTIPROCESSOR (SM)                                                                      |
|                                                                                                    |
|  +----------------------------------------------------------------------------------------------+  |
|  | L1 Instruction Cache (Shared across the 4 Sub-Cores)                                         |  |
|  +----------------------------------------------------------------------------------------------+  |
|                                                                                                    |
|  +---------------------------+ +---------------------------+ +----------------------------------+  |
|  | SUB-CORE 0                | | SUB-CORE 1                | | SUB-CORE 2 / 3                   |  |
|  |                           | |                           | |                                  |  |
|  | [ Warp Scheduler ]        | | [ Warp Scheduler ]        | | (Identical layout)               |  |
|  | [ Instruction Dispatch ]  | | [ Instruction Dispatch ]  | |                                  |  |
|  |                           | |                           | |                                  |  |
|  | [ Register File: 64 KB ]  | | [ Register File: 64 KB ]  | |                                  |  |
|  | (16,384 x 32-bit reg)     | | (16,384 x 32-bit reg)     | |                                  |  |
|  |                           | |                           | |                                  |  |
|  | [ 16 x FP32 Cores ]       | | [ 16 x FP32 Cores ]       | |                                  |  |
|  | [ 16 x INT32 Cores ]      | | [ 16 x INT32 Cores ]      | |                                  |  |
|  | [ 8 x FP64 Cores ]        | | [ 8 x FP64 Cores ]        | |                                  |  |
|  | [ 4 x SFUs ]              | | [ 4 x SFUs ]              | |                                  |  |
|  | [ 1 x 4th-Gen Tensor Core]| | [ 1 x 4th-Gen Tensor Core]| |                                  |  |
|  | [ 8 x Load/Store Units ]  | | [ 8 x Load/Store Units ]  | |                                  |  |
|  +---------------------------+ +---------------------------+ +----------------------------------+  |
|                                                                                                    |
|  +----------------------------------------------------------------------------------------------+  |
|  | COMBINED SHARED MEMORY / L1 DATA CACHE (Configurable SRAM: up to 228 KB per SM)              |  |
|  | - 32 independent banks (4 bytes wide per bank = 128 bytes/cycle throughput)                   |  |
|  +----------------------------------------------------------------------------------------------+  |
|  | TENSOR MEMORY ACCELERATOR (TMA) ENGINE (Asynchronous hardware copy: Global Memory <-> SRAM)   |  |
|  +----------------------------------------------------------------------------------------------+  |
+-------------------------------------------------------------------------------

## The difference between a CUDA Core and a Tensor Core:
 comes down to the granularity of math they are physically wired to perform in silicon: Scalar arithmetic vs. Matrix arithmetic.1. The Core Mechanical DifferenceCUDA Core (Scalar Math Engine)A standard CUDA Core is an Arithmetic Logic Unit (ALU). It operates on individual scalar numbers.Operation: 1 Fused Multiply-Add (FMA) per clock cycle:$$d = (a \times b) + c$$Each thread in a warp feeds its own scalar values ($a, b, c$) into its assigned CUDA Core to produce a single scalar result ($d$).Tensor Core (Matrix Math Engine)A Tensor Core is a specialized systolic-like array of hardwired multipliers and adders. It does not operate on single numbers; it operates on entire 2D matrix tiles.Operation: Matrix Multiply-Accumulate (MMA) in a single hardware instruction:$$\mathbf{D} = (\mathbf{A} \times \mathbf{B}) + \mathbf{C}$$A whole warp (32 threads) collaborates to feed chunks (fragments) of two matrix tiles ($\mathbf{A}$ and $\mathbf{B}$, typically $16 \times 16$) into the Tensor Core, which calculates the entire matrix product in silicon.CUDA CORE (Thread-Level Scalar ALU):
Thread 0 ──► [ a0 * b0 + c0 ] ──► d0  (1 scalar operation)
Thread 1 ──► [ a1 * b1 + c1 ] ──► d1  (1 scalar operation)

TENSOR CORE (Warp-Level Matrix Tile Engine):
All 32 Threads ──► [ Matrix A (16x16) × Matrix B (16x16) + Matrix C (16x16) ] ──► Matrix D (16x16)
                   (Computes hundreds to thousands of operations simultaneously)
2. Side-by-Side Architectural ComparisonDimensionStandard CUDA CoreTensor CoreInput Data TypeScalar values ($1 \times 1$)Matrix tiles (e.g., $16 \times 16$, $16 \times 8$)Execution Unit LevelExecuted per individual threadExecuted as a collective warp (32 threads)Supported PrecisionFP32, FP64, INT32Mixed-precision: FP16, BF16, FP8, INT8, INT4 (accumulating in FP32)Instruction Examplefma.rn.f32mma.sync.aligned.m16n8k16Math Efficiency1 Multiply-Add ($2\text{ FLOPs}$) per cycleHundreds to thousands of FLOPs per cycleBest Used ForElementwise ops (ReLU, GeLU, Add, LayerNorm, Softmax)Dense Matrix Multiplications (Linear layers, QKV projections, MLP/FFN)Throughput (e.g., NVIDIA H100)